# Project architecture, Hermes3 + RAG, and detailed workflow

This notebook expands the architecture overview and includes: 

- A shallow file-tree snapshot.
- A VS Code-friendly reference architecture with high-level definitions.
- Demo-safe, runnable walkthroughs for: student request → librarian approval → campaign winner selection,
  and the student return flow with a qualifying quiz.

It also documents where students and librarians can prompt Hermes3 (local LLM server) and how the RAG upload to Qdrant + embeddings fit into the flow. All code is guarded so the notebook opens without optional services installed.

In [ ]:
# 1) File structure snapshot (shallow)
import os, json
root = os.path.abspath(os.getcwd())
def tree(path, max_depth=2, _depth=0):
    if _depth > max_depth:
        return []
    items = []
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.name.startswith('.'):
                    continue
                if entry.is_dir():
                    items.append({'type':'dir','name':entry.name,'children':tree(os.path.join(path,entry.name),max_depth,_depth+1)})
                else:
                    items.append({'type':'file','name':entry.name})
    except PermissionError:
        pass
    return items
fs = tree(root, max_depth=2)
print('Root:', root)
print(json.dumps(fs, indent=2)[:8000])

## 2) VS Code reference architecture (high-level)

- `app/` — application code (UI builders, routers, LLM clients, RAG helpers, vector store).
- `scripts/` — command-line helpers and clients (batch jobs, migrations, client scripts for Hermes3).
- `docs/` — notebooks and developer docs (this folder).
- `tests/` — unit tests for critical flows.
- `requirements.txt` / `pyproject.toml` — dependency manifests.

Developer workflow (VS Code):
1) Open the repo folder in VS Code.
2) Create/select a virtual environment (`python3 -m venv .venv`) and select it in the Python extension.
3) Run the guarded notebooks in `docs/` to explore modules without installing heavy deps.
4) Use focused pytest runs and the Python REPL to iterate quickly.

High-level responsibilities (where to look):
- UI layers: `app/ui_student.py`, `app/ui_librarian.py`, `app/gradio_main.py`.
- LLM & local model client: `app/llm_client.py`, `app/models_llm.py` (Hermes3 client wrapper lives here / in `scripts/clients`).
- RAG / Retrieval: `app/rag.py`, `app/router.py`. Uploads invoke `upsert_chunks` → embeddings → Qdrant.
- Vector store abstraction: `app/vector_store.py` (wraps Qdrant client and exposes `upsert`, `search`, `stats`).
- Campaign & reporting: `app/campaign.py`, `app/report.py` (leaderboards, exports, weekly winner selection).

Key runtime pieces and purpose:
- Hermes3 (local LLM server): serve completions for timely responses (prompting from students/librarians).
- Qdrant (vector DB): store embeddings for RAG so librarian research answers are grounded in uploaded sources.
- Embeddings (e.g., BGE): convert text chunks into vectors for indexing and retrieval.

## 3) Hermes3 prompts — where and how students/librarians call it

- Students: use the *Learn more about a book* accordion to ask narrative questions (themes, characters, reading-level guidance). That request is routed through the `router` and may hit Hermes3 directly when the `route` is `llm`.
- Librarians: use the *Research Assistant* to synthesize web or vector hits; Hermes3 is used for synthesis or direct prompt completion. Librarian prompts may include system instructions (the `LIBRARIAN_SYSTEM_PROMPT`) and should be run with low temperature for deterministic answers.

Implementation notes:
- The project wraps Hermes3 with a small client `scripts/clients/complete.py` or `app/llm_client.py`. The client provides `completion(prompt, **kwargs)` and `embed(text)`.
- Prompts should be validated with `is_prompt_safe` before sending to Hermes3.

The cell below shows a guarded example of calling Hermes3 via the client. Replace `prompt` with a real prompt when your Hermes3 endpoint is configured.

In [ ]:
# Guarded Hermes3 completion example (demo-safe)
try:
    from scripts.clients import complete as hermes_client
    print('Found scripts.clients.complete (Hermes client)')
    prompt = 'Summarize why reading historical fiction helps students understand context.'
    try:
        out = hermes_client.complete(prompt, max_tokens=200)
        print('Hermes3 result (truncated):', str(out)[:1000])
    except Exception as e:
        print('Hermes client invocation failed:', e)
except Exception as e:
    print('Hermes client not available (ok in demo):', e)

## 4) Librarian RAG uploads → Qdrant (purpose & flow)

Purpose:

- Librarians can upload `.txt/.md/.csv/.json/.docx/.pdf` files to enrich the knowledge base.
- The upload flow extracts text, chunks it, embeds each chunk with an embeddings model, and upserts vectors into Qdrant for fast retrieval.

Key steps (implemented in `app.rag` and `app.vector_store`):
1) Read and sanitize files (`_read_file_by_path`).
2) Chunk text (`chunk_text`) into overlapping pieces for context preservation.
3) Create embeddings (`embed` via `llm_client` or embeddings client).
4) Upsert vectors into Qdrant using `vector_store.upsert(chunks)`.
5) Keep a lightweight `RAG_CORPUS` mapping for raw source fallback.

Why Qdrant?

- Qdrant stores dense vectors with payload metadata (source, chunk id). It supports efficient nearest-neighbor queries to surface context for librarian answers.
- It enables reproducible retrieval for RAG-based synthesis and auditability (you can show which source chunks grounded the answer).

In [ ]:
# Guarded RAG upload example (demo-safe)
try:
    import importlib
    rag = importlib.import_module('app.rag')
    vs = importlib.import_module('app.vector_store')
    print('Imported app.rag and app.vector_store')
    # Example: call rag.ui_rag_upload with a list of file paths (here we skip actual files)
    if hasattr(rag, 'ui_rag_upload'):
        print('ui_rag_upload available — demo example skipped (no files)')
    else:
        print('rag.ui_rag_upload not present — skip')
except Exception as e:
    print('RAG/vector modules not available:', e)

In [ ]:
## 5) Mock workflow: Student request → Librarian approval → Campaign winner selection → Return with quiz

The cells below execute a guarded simulation using the demo in-memory stores (from `app.gradio_main`) when available. The logic demonstrates: submission, librarian approval, campaign seeding, weekly winner selection, and returning a book with a qualifying quiz.


### 5.1 Demo-safe simulation (overview)

The cell below runs a guarded simulation using the in-memory demo stores provided by `app.gradio_main`.
If the module isn't available or the demo stores were not created, the cell will fall back to a lightweight local stub so you can still see the workflow logic.

This simulates:
- Student submits a book request (title, short reason).
- Librarian lists pending requests and approves one.
- The approved request is seeded into a campaign and a winner is chosen (deterministic selection for the demo).
- The student returns a book and takes a short qualifying quiz; pass/fail controls whether they get counted as completed.

In [ ]:
# 5.2 Demo-safe simulation (runnable)
import random, pprint
pprint = pprint.pprint
# Try to import the demo handlers from the app; fall back to stubs if missing
try:
    from app import gradio_main as gm
    print('Using app.gradio_main demo stores')
    # Helpers expected from the demo scaffold
    submit = getattr(gm, 'submit_book_request', None)
    list_pending = getattr(gm, 'librarian_list_pending', None)
    approve = getattr(gm, 'librarian_approve', None)
    refresh = getattr(gm, 'refresh_my_requests', None)
except Exception as e:
    print('app.gradio_main not available or import failed — using local stubs:', e)
    # Minimal local stubs that mimic expected behavior
    _STORES = {'BOOK_DB': {}, 'BOOK_REQUESTS': {}, 'LIBRARIAN_QUEUES': {}}
    _id_ctr = {'v': 1}
    def _make_id():
        _id = _id_ctr['v']; _id_ctr['v'] += 1; return str(_id)
    def submit(student, title, why):
        rid = _make_id()
        _STORES['BOOK_REQUESTS'][rid] = {'id': rid, 'student': student, 'title': title, 'why': why, 'status':'pending'}
        return rid
    def list_pending(librarian):
        return [v for v in _STORES['BOOK_REQUESTS'].values() if v['status']=='pending']
    def approve(librarian, request_id):
        r = _STORES['BOOK_REQUESTS'].get(request_id)
        if not r: return False
        r['status']='approved'; r['approved_by']=librarian; return True
    def refresh(student):
        return [v for v in _STORES['BOOK_REQUESTS'].values() if v['student']==student]
,
,
,
,
,
,
,
,
,
,

### 5.3 Notes and next steps

- The demo uses guarded imports so the notebook opens even when heavy services (Hermes3, Qdrant) are not running.
- Once your compose stack with Qdrant and Hermes3 is running, re-run the cells above to exercise the real upload/RAG flow.
- Next recommended checks: run the import smoke test from the repo root (import `app.gradio_main`, `app.rag`, `app.vector_store`, `scripts.clients.complete`).
- If you want, I can add an automated cell that runs a quick smoke-import and prints a short report; tell me if you'd like that inserted.